# Logistic Regression on Census Income Classification via BigQuery
**Task ID:** `logreg_lvl5_bq_income`  
**Series:** Logistic Regression, Level 5  
**Protocol:** `pytorch_task_v1`

## Overview
Binary classification on US Census income data loaded **from Google BigQuery**
(>50K vs <=50K) using `CrossEntropyLoss` with Adam optimizer. Includes manual
one-hot encoding of categorical features (workclass, marital_status, occupation).

## Math
- Cross-entropy loss: $L = -\frac{1}{N}\sum_i\left[y_i\log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$
- Softmax output (2 classes): $P(y=k|x) = \frac{e^{z_k}}{\sum_j e^{z_j}}$

In [ ]:
import sys
import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from google.cloud import bigquery

print(f"PyTorch version: {torch.__version__}")

## Required Functions (`pytorch_task_v1` protocol)

In [ ]:
def get_task_metadata():
    return {
        "task_id": "logreg_lvl5_bq_income",
        "algorithm": "Logistic Regression (BigQuery Census Income Classification)",
        "series": "Logistic Regression",
        "level": 5,
        "interface_protocol": "pytorch_task_v1",
    }

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

print(f"Device: {get_device()}")
print(f"Task: {get_task_metadata()['task_id']}")

## Load Data from BigQuery

**Table:** `bigquery-public-data.ml_datasets.census_adult_income`

We pull both numeric and categorical columns via SQL, then one-hot encode
the categoricals manually in Python (no sklearn `OneHotEncoder`).

In [ ]:
GCP_PROJECT_ID = "gen-lang-client-0916541599"

def load_data_from_bigquery(project_id=GCP_PROJECT_ID):
    """Query Census Adult Income data from BigQuery public dataset."""
    client = bigquery.Client(project=project_id)
    query = """
    SELECT
        age,
        workclass,
        education_num,
        marital_status,
        occupation,
        hours_per_week,
        capital_gain,
        capital_loss,
        income_bracket
    FROM `bigquery-public-data.ml_datasets.census_adult_income`
    WHERE income_bracket IS NOT NULL
    """
    df = client.query(query).to_dataframe()
    print(f"Loaded {len(df)} rows from BigQuery table: census_adult_income")
    print(f"Columns: {list(df.columns)}")
    print(f"Income distribution:\n{df['income_bracket'].value_counts()}")
    return df

# --- ARCHIVED: sklearn fallback (same UCI dataset, no BigQuery needed) ---
# def load_data_fallback():
#     from sklearn.datasets import fetch_openml
#     adult = fetch_openml('adult', version=2, as_frame=True)
#     df = adult.frame[['age', 'workclass', 'education-num', 'marital-status',
#                        'occupation', 'hours-per-week', 'capital-gain',
#                        'capital-loss', 'class']].copy()
#     df.columns = ['age', 'workclass', 'education_num', 'marital_status',
#                   'occupation', 'hours_per_week', 'capital_gain',
#                   'capital_loss', 'income_bracket']
#     df = df.dropna()
#     df['income_bracket'] = df['income_bracket'].map(
#         lambda x: '>50K' if '>50K' in str(x) else '<=50K')
#     return df

In [ ]:
def manual_one_hot(series):
    """One-hot encode a pandas Series manually (no sklearn)."""
    categories = sorted(series.unique())
    cat_to_idx = {c: i for i, c in enumerate(categories)}
    n = len(series)
    k = len(categories)
    encoded = np.zeros((n, k), dtype=np.float32)
    for i, val in enumerate(series):
        encoded[i, cat_to_idx[val]] = 1.0
    return encoded, categories

def make_dataloaders(cfg):
    df = load_data_from_bigquery()

    # Numeric features
    numeric_cols = ['age', 'education_num', 'hours_per_week', 'capital_gain', 'capital_loss']
    X_num = df[numeric_cols].values.astype(np.float32)

    # One-hot encode categorical features manually
    cat_cols = ['workclass', 'marital_status', 'occupation']
    cat_arrays = []
    all_feature_names = list(numeric_cols)
    for col in cat_cols:
        encoded, categories = manual_one_hot(df[col].astype(str))
        cat_arrays.append(encoded)
        all_feature_names.extend([f"{col}_{c}" for c in categories])
        print(f"  {col}: {len(categories)} categories -> {len(categories)} one-hot columns")

    X_all = np.hstack([X_num] + cat_arrays)

    # Binary target: >50K = 1, <=50K = 0
    y_np = (df['income_bracket'].str.contains('>50K')).astype(np.int64).values

    # Train/val split
    n = len(X_all)
    n_val = int(n * cfg.get("val_ratio", 0.2))
    idx = np.random.permutation(n)
    train_idx, val_idx = idx[n_val:], idx[:n_val]

    # Standardize numeric features only (using train stats)
    n_num = len(numeric_cols)
    X_mean = X_all[train_idx, :n_num].mean(axis=0)
    X_std = X_all[train_idx, :n_num].std(axis=0) + 1e-8
    X_all[:, :n_num] = (X_all[:, :n_num] - X_mean) / X_std

    X = torch.from_numpy(X_all)
    y = torch.from_numpy(y_np)

    train_ds = TensorDataset(X[train_idx], y[train_idx])
    val_ds = TensorDataset(X[val_idx], y[val_idx])

    input_dim = X_all.shape[1]
    print(f"\nTrain: {len(train_ds)}, Val: {len(val_ds)}")
    print(f"Features: {input_dim} ({n_num} numeric + {input_dim - n_num} one-hot)")

    return (
        DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True),
        DataLoader(val_ds, batch_size=cfg["batch_size"]),
        {"input_dim": input_dim, "feature_names": all_feature_names},
    )

In [ ]:
def build_model(cfg):
    return nn.Linear(cfg["input_dim"], 2)  # 2 classes: <=50K, >50K

def train(model, train_loader, val_loader, cfg, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])
    epochs = cfg["epochs"]
    loss_history = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(X_b)
        loss_history.append(epoch_loss / len(train_loader.dataset))

        if (epoch + 1) % 20 == 0:
            model.eval()
            val_loss, correct, total = 0.0, 0, 0
            with torch.no_grad():
                for X_b, y_b in val_loader:
                    X_b, y_b = X_b.to(device), y_b.to(device)
                    logits = model(X_b)
                    val_loss += criterion(logits, y_b).item() * len(X_b)
                    correct += (logits.argmax(1) == y_b).sum().item()
                    total += len(y_b)
            val_loss /= total
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {loss_history[-1]:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {correct/total:.4f}")

    return loss_history

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b = X_b.to(device)
            logits = model(X_b).cpu()
            all_preds.append(logits.argmax(1))
            all_targets.append(y_b)
    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)

    accuracy = (preds == targets).float().mean().item()

    # Confusion matrix
    tp = ((preds == 1) & (targets == 1)).sum().item()
    fp = ((preds == 1) & (targets == 0)).sum().item()
    fn = ((preds == 0) & (targets == 1)).sum().item()
    tn = ((preds == 0) & (targets == 0)).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return {
        "accuracy": round(accuracy, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "confusion_matrix": {"tp": tp, "fp": fp, "fn": fn, "tn": tn},
    }

def predict(model, X, device):
    model.eval()
    with torch.no_grad():
        logits = model(X.to(device)).cpu()
        return logits.argmax(1)

def save_artifacts(outputs, cfg):
    out_dir = cfg.get("output_dir", "./output")
    os.makedirs(out_dir, exist_ok=True)
    saveable = {k: v for k, v in outputs.items() if isinstance(v, (int, float, str, dict, list))}
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(saveable, f, indent=2, default=str)
    print(f"Metrics saved to {out_dir}")

## Configuration

In [ ]:
cfg = {
    "batch_size": 128,
    "val_ratio": 0.2,
    "epochs": 100,
    "lr": 0.001,
    "accuracy_threshold": 0.80,
    "output_dir": "./output/logreg_lvl5_bq_income",
}

## Train

In [ ]:
set_seed(42)
device = get_device()

train_loader, val_loader, data_info = make_dataloaders(cfg)
cfg["input_dim"] = data_info["input_dim"]
model = build_model(cfg).to(device)

print(f"\nModel: nn.Linear({data_info['input_dim']}, 2)")
print("--- Training ---")
loss_history = train(model, train_loader, val_loader, cfg, device)

## Evaluate

In [ ]:
train_metrics = evaluate(model, train_loader, device)
val_metrics = evaluate(model, val_loader, device)

print(f"Train | Acc: {train_metrics['accuracy']:.4f} | P: {train_metrics['precision']:.4f} | R: {train_metrics['recall']:.4f} | F1: {train_metrics['f1']:.4f}")
print(f"Val   | Acc: {val_metrics['accuracy']:.4f} | P: {val_metrics['precision']:.4f} | R: {val_metrics['recall']:.4f} | F1: {val_metrics['f1']:.4f}")

cm = val_metrics["confusion_matrix"]
print(f"\nConfusion Matrix (Val):")
print(f"              Predicted <=50K  Predicted >50K")
print(f"  Actual <=50K    {cm['tn']:>6}         {cm['fp']:>6}")
print(f"  Actual >50K     {cm['fn']:>6}         {cm['tp']:>6}")

## Save Artifacts & Quality Assertions

In [ ]:
outputs = {
    "loss_history": loss_history,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
    "feature_names": data_info["feature_names"],
}
save_artifacts(outputs, cfg)

print("\n" + "=" * 60)
print("Quality Assertions...")
print("=" * 60)

assert val_metrics["accuracy"] >= cfg["accuracy_threshold"], \
    f"Val accuracy {val_metrics['accuracy']:.4f} < {cfg['accuracy_threshold']}"
print(f"PASS: Val accuracy {val_metrics['accuracy']:.4f} >= {cfg['accuracy_threshold']}")

assert abs(train_metrics["accuracy"] - val_metrics["accuracy"]) < 0.05, \
    f"Overfitting: train_acc={train_metrics['accuracy']:.4f} vs val_acc={val_metrics['accuracy']:.4f}"
print(f"PASS: No overfitting (train-val gap: {abs(train_metrics['accuracy'] - val_metrics['accuracy']):.4f})")

print("\n=== ALL CHECKS PASSED ===")
exit_code = 0
print(f"sys.exit({exit_code})")